# Exploración de las viviendas de Barcelona

Usad este notebook para conocer los datos **antes** de escribir `SPECS.md`. Lo que encontréis va a la sección **Datos de entrada y problemas** de `SPECS.md`, y las ideas para tratarlo, a **Estrategias y experimentos**.

Aquí se mira y se piensa: nada de lo que hagáis en este notebook entra en el modelo. Las transformaciones que decidáis se escriben en `SPECS.md`, y el agente las implementa en `src/`, dentro del `Pipeline` (o en `limpiar_objetivo()`, si son sobre el precio), para que se apliquen igual en la validación cruzada, en la validación y en el test. Este notebook es el único sitio donde escribís código vosotros.

El significado de cada columna está en `DATOS.md`. Recordad: los barrios son reales, pero las viviendas y los precios son simulados.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px

sys.path.insert(0, str(Path.cwd().parent))  # para importar src/ desde notebooks/

from src.datos import cargar_entrenamiento, cargar_test

entrenamiento = cargar_entrenamiento()
test = cargar_test()
print(f"Entrenamiento: {entrenamiento.shape[0]:,d} filas · Test: {test.shape[0]:,d} filas")

Entrenamiento: 6,237 filas · Test: 4,002 filas


## 1. Perfil de cada columna

In [2]:
def perfil(df: pd.DataFrame) -> pd.DataFrame:
    """Tipo, nulos, ceros, valores distintos y un ejemplo de cada columna."""
    return pd.DataFrame({
        "tipo": df.dtypes.astype(str),
        "nulos": df.isna().sum(),
        "% nulos": (df.isna().mean() * 100).round(1),
        "ceros": (df == 0).sum(),
        "distintos": df.nunique(),
        "ejemplo": df.iloc[0],
    })

perfil(entrenamiento)

,tipo,nulos,% nulos,ceros,distintos,ejemplo
fecha_venta,str,0,0.0,0,1355,17/02/2024
municipi,str,0,0.0,0,12,Barcelona
districte,str,1503,24.1,0,10,Sant Andreu
barri,str,1503,24.1,0,72,el Bon Pastor
direccion,str,183,2.9,0,4992,"Passeig de l'Havana, 16"
codigo_postal,float64,218,3.5,0,75,8030.0
latitud,float64,108,1.7,0,3225,41.43817
longitud,float64,108,1.7,0,3253,2.19852
distancia_centro_km,float64,0,0.0,0,161,6.2
ref_catastral,str,0,0.0,0,6084,9130140CM9461Z1396MC


**¿Pasa lo mismo en el test?** El modelo tendrá que tasar esas 4.002 viviendas, con sus nulos y sus rarezas.

In [3]:
pd.DataFrame({
    "% nulos entrenamiento": (entrenamiento.isna().mean() * 100).round(1),
    "% nulos test": (test.isna().mean() * 100).round(1),
}).dropna(how="all").query("`% nulos entrenamiento` > 0 or `% nulos test` > 0")

,% nulos entrenamiento,% nulos test
any_construccio,29.8,30.1
ascensor,15.0,14.7
barri,24.1,23.1
certificado_energetico,8.1,8.4
codigo_postal,3.5,3.0
direccion,2.9,2.8
districte,24.1,23.1
estado,6.8,7.3
latitud,1.7,1.6
longitud,1.7,1.6


## 2. La variable objetivo

In [4]:
px.histogram(entrenamiento, x="precio", nbins=100, template="simple_white",
             title="Precio de la operación", labels={"precio": "Precio (€)"})

ValueError: Mime type rendering requires nbformat>=4.2.0 but it is not installed

Figure({
    'data': [{'bingroup': 'x',
              'hovertemplate': 'Precio (€)=%{x}<br>count=%{y}<extra></extra>',
              'legendgroup': '',
              'marker': {'color': '#1F77B4', 'pattern': {'shape': ''}},
              'name': '',
              'nbinsx': 100,
              'orientation': 'v',
              'showlegend': False,
              'type': 'histogram',
              'x': {'bdata': ('AAAAAABXB0EAAAAAaK0sQQAAAACYwy' ... 'AAAJDnFEEAAAAA8LIWQQAAAABAxxZB'),
                    'dtype': 'f8'},
              'xaxis': 'x',
              'yaxis': 'y'}],
    'layout': {'barmode': 'relative',
               'legend': {'tracegroupgap': 0},
               'template': '...',
               'title': {'text': 'Precio de la operación'},
               'xaxis': {'anchor': 'y', 'domain': [0.0, 1.0], 'title': {'text': 'Precio (€)'}},
               'yaxis': {'anchor': 'x', 'domain': [0.0, 1.0], 'title': {'text': 'count'}}}
})

- ¿Qué forma tiene la distribución? ¿Cuáles son los precios más bajos y los más altos? ¿Alguno es imposible?
- Repetid el histograma con el logaritmo del precio. ¿Qué cambia? ¿Qué implicaría para un modelo lineal?
- `estado_operacion` dice si la operación se cerró. ¿Qué significa el precio de un anuncio **retirado**? ¿Es comparable con el de una venta?

In [ ]:
entrenamiento.groupby("estado_operacion").agg(
    filas=("precio", "size"),
    precio_mediano=("precio", "median"),
    euros_m2_mediano=("precio", lambda p: (p / entrenamiento.loc[p.index, "superficie_construida"].replace(0, np.nan)).median()),
)

## 3. Preguntas de calidad para empezar

- ¿Hay superficies imposibles (0, enormes)? ¿Es siempre la construida mayor que la útil?
- ¿Qué valores toma `planta`? ¿Cómo la convertiríais en un número? ¿Qué es "Entlo" o "Pral"?
- ¿Qué valores toman `ascensor`, `parking` y `tipo`? ¿Cuántas formas distintas hay de decir lo mismo? ¿Qué significa "Opcional" en `parking`?
- ¿Qué significa un 0 en `terraza_m2`? ¿Y un vacío?
- ¿Hay años de construcción imposibles?
- ¿En qué formatos está `fecha_venta`?
- ¿Cuándo falta el barrio? ¿Siempre fuera de Barcelona?
- ¿Hay `ref_catastral` repetidas? ¿Es la misma operación dos veces o la misma vivienda vendida dos veces?
- `dias_en_venta`: ¿lo sabe el banco en el momento de tasar?
- ¿Cuántas operaciones tiene cada barrio y cada municipio? ¿Hay zonas del test que no aparezcan en el entrenamiento?

**Ejemplo:** los diez valores más frecuentes de `planta`

In [5]:
entrenamiento["planta"].value_counts(dropna=False).head(10)

planta
NaN          302
3º           248
2º           243
1º           231
4º           227
5º           217
Bajo         211
Ático        158
6º           143
1ª planta    139
Name: count, dtype: int64

**Ejemplo:** zonas del test que no están en el entrenamiento

In [7]:
viviendas_nuevas = test.loc[nuevas, ["municipi", "barri", "districte", "direccion"]]
viviendas_nuevas

,municipi,barri,districte,direccion
index,,,,
3167,Barcelona,la Clota,Horta-Guinardó,"Camí Sant Genís a Horta, 58"
3328,Barcelona,la Clota,Horta-Guinardó,"Camí de Sant Genís a Horta, 58"
4350,Barcelona,la Clota,Horta-Guinardó,camí sant genís a horta 58
6007,Barcelona,la Clota,Horta-Guinardó,"Camino Sant Genís a Horta, 58"


In [6]:
zona_ent = entrenamiento["barri"].fillna(entrenamiento["municipi"])
zona_test = test["barri"].fillna(test["municipi"])
nuevas = ~zona_test.isin(zona_ent)
print(f"{nuevas.sum()} viviendas del test están en {zona_test[nuevas].nunique()} zonas sin operaciones en el entrenamiento")

4 viviendas del test están en 1 zonas sin operaciones en el entrenamiento


In [12]:
datos = pd.concat([entrenamiento, test], ignore_index=True)

for columna in ["parking"]:
    valores = datos[columna].astype("string").str.strip()
    normalizados = valores.str.lower()

    if columna in ["parking"]:
        normalizados = normalizados.replace({
            "sí": "si",
            "true": "si",
            "1": "si",
            "false": "no",
            "0": "no",
        })

    resumen = (
        pd.DataFrame({"normalizado": normalizados, "valor_original": valores})
        .dropna(subset=["normalizado"])
        .groupby("normalizado")
        .agg(
            formas=("valor_original", lambda x: sorted(x.unique().tolist())),
            cantidad_formas=("valor_original", "nunique"),
            filas=("valor_original", "size"),
        )
        .sort_values("filas", ascending=False)
    )

    print(f"\n{columna}: valores originales")
    print(valores.value_counts(dropna=False).to_string())
    print(f"\n{columna}: formas agrupadas")
    print(resumen.to_string())


parking: valores originales
parking
No          6552
Sí          1325
<NA>        1024
Opcional     712
Incluido     626

parking: formas agrupadas
                   formas  cantidad_formas  filas
normalizado                                      
no                 ['No']                1   6552
si                 ['Sí']                1   1325
opcional     ['Opcional']                1    712
incluido     ['Incluido']                1    626


In [ ]:
# Vuestras comprobaciones aquí

## 4. Relación con el precio

Antes de probar una transformación, mirad cómo se relaciona cada variable con el precio. Una relación curva, valores imposibles o unos pocos puntos muy alejados son pistas de qué estrategia probar.

In [8]:
px.scatter(entrenamiento, x="superficie_construida", y="precio", color="municipi", opacity=0.4,
           template="simple_white", title="Precio frente a superficie construida",
           labels={"superficie_construida": "Superficie construida (m²)", "precio": "Precio (€)"})

ValueError: Mime type rendering requires nbformat>=4.2.0 but it is not installed

Figure({
    'data': [{'hovertemplate': ('municipi=Barcelona<br>Superfic' ... 'Precio (€)=%{y}<extra></extra>'),
              'legendgroup': 'Barcelona',
              'marker': {'color': '#1F77B4', 'opacity': 0.4, 'symbol': 'circle'},
              'mode': 'markers',
              'name': 'Barcelona',
              'showlegend': True,
              'type': 'scattergl',
              'x': {'bdata': ('AAAAAACARkAAAAAAAGBjQAAAAAAAQG' ... 'AAAIBYQAAAAAAAQFtAAAAAAAAATEA='),
                    'dtype': 'f8'},
              'xaxis': 'x',
              'y': {'bdata': ('AAAAAABXB0EAAAAAaK0sQQAAAACYwy' ... 'AAAKQPQQAAAADABBNBAAAAAJDnFEE='),
                    'dtype': 'f8'},
              'yaxis': 'y'},
             {'hovertemplate': ('municipi=Badalona<br>Superfici' ... 'Precio (€)=%{y}<extra></extra>'),
              'legendgroup': 'Badalona',
              'marker': {'color': '#FF7F0E', 'opacity': 0.4, 'symbol': 'circle'},
              'mode': 'markers',
              'name': 'Badalona',
              'showlegend': True,
              'type': 'scattergl',
              'x': {'bdata': ('AAAAAABAUkAAAAAAAEBaQAAAAAAAwF' ... 'AAAMBVQAAAAAAAAFhAAAAAAACASEA='),
                    'dtype': 'f8'},
              'xaxis': 'x',
              'y': {'bdata': ('AAAAAADW/0AAAAAAYJcSQQAAAACAAQ' ... 'AA4AcDQQAAAACgXwRBAAAAAACP9kA='),
                    'dtype': 'f8'},
              'yaxis': 'y'},
             {'hovertemplate': ('municipi=Sant Cugat del Vallès' ... 'Precio (€)=%{y}<extra></extra>'),
              'legendgroup': 'Sant Cugat del Vallès',
              'marker': {'color': '#2CA02C', 'opacity': 0.4, 'symbol': 'circle'},
              'mode': 'markers',
              'name': 'Sant Cugat del Vallès',
              'showlegend': True,
              'type': 'scattergl',
              'x': {'bdata': ('AAAAAACgZ0AAAAAAACBhQAAAAAAA4G' ... 'AAAAAAQ0AAAAAAAMBkQAAAAAAAIG1A'),
                    'dtype': 'f8'},
              'xaxis': 'x',
              'y': {'bdata': ('AAAAANhHLEEAAAAAeLQpQQAAAAAwbS' ... 'AAAGDPAUEAAAAAaIwlQQAAAAA4AzBB'),
                    'dtype': 'f8'},
              'yaxis': 'y'},
             {'hovertemplate': ('municipi=Santa Coloma de Grame' ... 'Precio (€)=%{y}<extra></extra>'),
              'legendgroup': 'Santa Coloma de Gramenet',
              'marker': {'color': '#D62728', 'opacity': 0.4, 'symbol': 'circle'},
              'mode': 'markers',
              'name': 'Santa Coloma de Gramenet',
              'showlegend': True,
              'type': 'scattergl',
              'x': {'bdata': ('AAAAAADAV0AAAAAAAABeQAAAAAAAgE' ... 'AAAAAAYUAAAAAAAIBXQAAAAAAAwFFA'),
                    'dtype': 'f8'},
              'xaxis': 'x',
              'y': {'bdata': ('AAAAAABWA0EAAAAA4NsBQQAAAADA+P' ... 'AAAECdDUEAAAAA4PUFQQAAAADAewRB'),
                    'dtype': 'f8'},
              'yaxis': 'y'},
             {'hovertemplate': ('municipi=El Prat de Llobregat<' ... 'Precio (€)=%{y}<extra></extra>'),
              'legendgroup': 'El Prat de Llobregat',
              'marker': {'color': '#9467BD', 'opacity': 0.4, 'symbol': 'circle'},
              'mode': 'markers',
              'name': 'El Prat de Llobregat',
              'showlegend': True,
              'type': 'scattergl',
              'x': {'bdata': ('AAAAAABAVkAAAAAAAEBZQAAAAAAAQF' ... 'AAAACAU0AAAAAAAIBYQAAAAAAAQFJA'),
                    'dtype': 'f8'},
              'xaxis': 'x',
              'y': {'bdata': ('AAAAAADHBUEAAAAAYCYQQQAAAACAgR' ... 'AAACAvCkEAAAAAgJQTQQAAAADgLQVB'),
                    'dtype': 'f8'},
              'yaxis': 'y'},
             {'hovertemplate': ('municipi=Esplugues de Llobrega' ... 'Precio (€)=%{y}<extra></extra>'),
              'legendgroup': 'Esplugues de Llobregat',
              'marker': {'color': '#8C564B', 'opacity': 0.4, 'symbol': 'circle'},
              'mode': 'markers',
              'name': 'Esplugues de Llobregat',
              'showlegend': True,
              'type': 'scattergl',
              'x': {'bdata': ('AAAAA

- ¿Qué le pasa a este gráfico con ejes logarítmicos (`log_x=True, log_y=True`)?
- Repetidlo coloreando por `tipo` o por `estado`.
- Cread una columna con los €/m² y comparadla entre barrios: ¿cuánto varía?

In [9]:
entrenamiento.select_dtypes("number").corr(method="spearman")["precio"].sort_values()

distancia_centro_km     -0.400045
codigo_postal           -0.341811
longitud                -0.225743
latitud                 -0.219132
dias_en_venta           -0.040259
any_construccio         -0.029850
poblacion_zona           0.018046
terraza_m2               0.165615
banos                    0.507856
habitaciones             0.593337
superficie_construida    0.702580
superficie_util          0.713250
precio                   1.000000
Name: precio, dtype: float64

- ¿Por qué usamos la correlación de Spearman y no la de Pearson? (Sesión 06.)
- Una correlación baja, ¿quiere decir que la variable no sirve para tasar?

## 5. La ubicación

In [ ]:
con_coordenadas = entrenamiento.dropna(subset=["latitud", "longitud"])
px.scatter_map(con_coordenadas, lat="latitud", lon="longitud", color=np.log10(con_coordenadas["precio"].clip(lower=1e4)),
               opacity=0.6, zoom=10, height=600, map_style="carto-positron",
               title="Precio según la ubicación", labels={"color": "log₁₀ precio"})

- ¿Dónde están las viviendas caras? ¿Ves algún punto fuera de sitio?
- La ubicación está en varias columnas: `municipi`, `districte`, `barri`, las coordenadas y `distancia_centro_km`. ¿Cuál usaríais en un modelo lineal y por qué?

## 6. Ideas para características nuevas

Pensad como un tasador: ¿qué miraría que no esté directamente en una columna? Algunas pistas: la altura real de la vivienda, si hay que subir escaleras, la época del edificio, el momento de la venta.

Escribid cada idea en `SPECS.md` §4 con su hipótesis **antes** de probarla.

In [ ]:
# Vuestras ideas aquí